# Table of Contents
* [Laptop Price Regression Model](#intro)
* [Importing Libraries and Data](#import)
* [Exploratory Data Analysis](#eda)
* [Normality Test](#stats)
* [Splitting Data](#train_split)
* [Preprocessing Data](#preprocessing)
* [Setting Model](#set)
* [Training Model](#training)
* [Best Results](#compare_results)
* [Finalizing Workflow](#workflow)
* [Fitting the final model](#fit)
* [API (FastAPI)](#api)
* [Interface(Streamlit)](#interface)
* [Automation(Docker)](#auto)
* [Saving Files](#store)
* [Conclusion](#conclusion)

## Machine Predictive Maintenance Classification Model <a class="anchor" id="intro"></a>
Introduction

## Importing Libraries and Data<a id="import"></a>

In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import explained_variance_score,mean_absolute_error,r2_score
from time import time
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import ExtraTreesRegressor
import pickle
import json

# read in all our data
laptop_train = pd.read_csv("./../data/raw/laptops_train.csv")
laptop_test = pd.read_csv("./../data/raw/laptops_test.csv")

# set seed for reproducibility
np.random.seed(0)

In [3]:
frames = [laptop_train,laptop_test]
df = pd.concat(frames)

In [4]:
df.head()

,Manufacturer,Model Name,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Operating System Version,Weight,Price
0,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,NaN,1.37kg,11912523.48
1,Apple,Macbook Air,Ultrabook,"13.3""",1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,NaN,1.34kg,7993374.48
2,HP,250 G6,Notebook,"15.6""",Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,NaN,1.86kg,5112900.00
3,Apple,MacBook Pro,Ultrabook,"15.4""",IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,NaN,1.83kg,22563005.40
4,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,NaN,1.37kg,16037611.20


## Exploratory Data Analysis<a id="eda"></a>

## Preprocessing Data<a id="preprocessing"></a>

In [5]:
# get the number of missing data points per column
missing_values_count = df.isnull().sum()
missing_values_count

Manufacturer                  0
Model Name                    0
Category                      0
Screen Size                   0
Screen                        0
CPU                           0
RAM                           0
 Storage                      0
GPU                           0
Operating System              0
Operating System Version    170
Weight                        0
Price                         0
dtype: int64

In [6]:
df['Operating System Version'].value_counts()

Operating System Version
10      1071
7         45
X          8
10 S       8
Name: count, dtype: int64

In [7]:
df[df['Operating System Version'].isna()].groupby('Operating System').groups

{'Android': [50, 136], 'Chrome OS': [290, 317, 430, 437, 472, 504, 584, 619, 677, 690, 697, 745, 762, 817, 828, 838, 846, 888, 907, 949, 953, 959, 71, 110, 124, 213, 237], 'Linux': [36, 40, 42, 59, 69, 96, 97, 102, 180, 210, 220, 235, 236, 272, 276, 281, 296, 338, 340, 379, 394, 403, 415, 423, 427, 439, 446, 524, 543, 555, 579, 604, 612, 613, 614, 616, 651, 669, 711, 712, 728, 769, 832, 852, 876, 886, 896, 965, 112, 145, 204, 210, 220, 249, 264, 272, 285, 289, 299, 303, 313, 317], 'No OS': [2, 10, 11, 18, 22, 46, 62, 76, 122, 125, 136, 138, 142, 172, 184, 187, 192, 193, 205, 212, 213, 216, 242, 261, 267, 279, 289, 303, 305, 356, 364, 367, 381, 463, 509, 514, 527, 547, 594, 627, 643, 654, 704, 725, 752, 783, 850, 859, 863, 871, 932, 947, 14, 78, 83, 113, 150, 172, 174, 186, 195, 196, 218, 223, 239, 270], 'macOS': [0, 1, 3, 4, 7, 12, 14, 15, 17, 45, 81, 249, 270]}

In [8]:
grouped = df.groupby('Operating System')

for name,group in grouped:
    print(name)
    print(group['Operating System Version'].unique())

Android
[nan]
Chrome OS
[nan]
Linux
[nan]
Mac OS
['X']
No OS
[nan]
Windows
['10' '10 S' '7']
macOS
[nan]


In [9]:
#Drop the Operating System Version because the laptop can just be updated by the user
df = df.drop('Operating System Version', axis=1)

In [10]:
df.columns

Index(['Manufacturer', 'Model Name', 'Category', 'Screen Size', 'Screen',
       'CPU', 'RAM', ' Storage', 'GPU', 'Operating System', 'Weight', 'Price'],
      dtype='object')

In [11]:
df[['Screen Size','Screen']]

,Screen Size,Screen
0,"13.3""",IPS Panel Retina Display 2560x1600
1,"13.3""",1440x900
2,"15.6""",Full HD 1920x1080
3,"15.4""",IPS Panel Retina Display 2880x1800
4,"13.3""",IPS Panel Retina Display 2560x1600
...,...,...
320,"14.0""",IPS Panel Full HD / Touchscreen 1920x1080
321,"13.3""",IPS Panel Quad HD+ / Touchscreen 3200x1800
322,"14.0""",1366x768
323,"15.6""",1366x768


In [12]:
#Data Preprocessing
#Course of Action:
#Manufacturer         Target Encoding
#Model Name           Split on the () into two columns- Target Encoding
#Category             One Hot Encoding
#Screen Size          Convert to float and scale (units is inches)
#Screen               Split into new features - screen type, screen quality, HD(Binary), Touchscreen(Binary)
#CPU                  Split into new features - Brand, Model Number, Speed
#RAM                  Remove "GB" and scale
#Storage              Convert to Total Storage column and scale
#GPU                  Split into new features - Brand, Model Number, Speed
#Operating System     One Hot Encoding (macOS = MacOS)
#Weight               Convert to lbs and scale (4s = 4.04 typo)
#Price                Convert to USD

In [13]:
#Treat Manufacturer as an nominal variable
df['Manufacturer'].unique()

array(['Apple', 'HP', 'Acer', 'Asus', 'Dell', 'Lenovo', 'Chuwi', 'MSI',
       'Microsoft', 'Toshiba', 'Huawei', 'Xiaomi', 'Vero', 'Razer',
       'Mediacom', 'Samsung', 'Google', 'Fujitsu', 'LG'], dtype=object)

In [14]:
# gives a tuple of column name and series
# for each column in the dataframe
for (columnName, columnData) in df.items():
    print('Column Name : ', columnName)
    print('Column Contents : ', columnData.unique())

Column Name :  Manufacturer
Column Contents :  ['Apple' 'HP' 'Acer' 'Asus' 'Dell' 'Lenovo' 'Chuwi' 'MSI' 'Microsoft'
 'Toshiba' 'Huawei' 'Xiaomi' 'Vero' 'Razer' 'Mediacom' 'Samsung' 'Google'
 'Fujitsu' 'LG']
Column Name :  Model Name
Column Contents :  ['MacBook Pro' 'Macbook Air' '250 G6' 'Aspire 3' 'ZenBook UX430UN'
 'Swift 3' 'Inspiron 3567' 'MacBook 12"' 'IdeaPad 320-15IKB' 'XPS 13'
 'Vivobook E200HA' 'Legion Y520-15IKBN' '255 G6' 'Inspiron 5379'
 '15-BS101nv (i7-8550U/8GB/256GB/FHD/W10)' 'MacBook Air' 'Inspiron 5570'
 'Latitude 5590' 'ProBook 470' 'LapBook 15.6"'
 'E402WA-GA010T (E2-6110/2GB/32GB/W10)'
 '17-ak001nv (A6-9220/4GB/500GB/Radeon' 'IdeaPad 120S-14IAP'
 'Inspiron 5770' 'ProBook 450' 'X540UA-DM186 (i3-6006U/4GB/1TB/FHD/Linux)'
 'Inspiron 7577' 'X542UQ-GO005 (i5-7200U/8GB/1TB/GeForce'
 'Aspire A515-51G' 'Inspiron 7773' 'IdeaPad 320-15ISK' 'Rog Strix'
 'X751NV-TY001T (N4200/4GB/1TB/GeForce' 'Yoga Book' 'ProBook 430'
 'Inspiron 3576' '15-bs002nv (i3-6006U/4GB/128GB/FHD/W10)'

In [15]:
df.dtypes

Manufacturer         object
Model Name           object
Category             object
Screen Size          object
Screen               object
CPU                  object
RAM                  object
 Storage             object
GPU                  object
Operating System     object
Weight               object
Price               float64
dtype: object

In [ ]:
model_name = df['Model Name'].unique()
model_name.sort()
model_name

In [ ]:
df[['Model Name','CPU']]

In [ ]:
df['Model Name'].str.contains('/')

In [ ]:
print('Yoga' in df['Model Name'].unique())

In [ ]:
df[df['Model Name'].str.contains('/')]

In [ ]:
category = df['Category'].unique()
category.sort()
category

In [ ]:
# gives a tuple of column name and series
# for each column in the dataframe
for (columnName, columnData) in df.items():
    print('Column Name : ', columnName)
    print('Column Contents : ', columnData.nunique())

In [ ]:
screen_size = df['Screen Size'].unique()
screen_size.sort()
screen_size

In [ ]:
screen = df['Screen'].unique()
screen.sort()
print(df['Screen'].value_counts())
screen

In [ ]:
CPU = df['CPU'].unique()
CPU.sort()
print(df['CPU'].value_counts())
CPU

In [ ]:
RAM = df['RAM'].unique()
RAM.sort()
print(df['RAM'].value_counts())
RAM

In [ ]:
storage = df[' Storage'].unique()
storage.sort()
print(df[' Storage'].value_counts())
storage

In [ ]:
GPU = df['GPU'].unique()
GPU.sort()
print(df['GPU'].value_counts())
GPU

In [ ]:
OS = df['Operating System'].unique()
OS.sort()
print(df['Operating System'].value_counts())
OS

In [ ]:
weight = df['Weight'].unique()
weight.sort()
print(df['Weight'].value_counts())
weight

In [ ]:
price = df['Price'].unique()
price.sort()
print(df['Price'].value_counts())
price

In [ ]:
df.columns

In [ ]:
#1 INR = 0.012203 USD Conversion Rate as of May 10,2023
df['Price_USD'] = df.Price/81.9433
df.Price_USD

In [ ]:
df[['Price','Price_USD']]

In [ ]:
#Target Encoding for Manfacturer


In [ ]:
df['Weight_LBS'] = df.Weight*2.204623
df.Weight_LBS

In [ ]:
df.Weight = df.Weight.str.replace('kg','')

In [ ]:
df.Weight = df.Weight.astype(float)

In [ ]:
#4s is a typo. Google search the weight of the laptop
df.Weight = df.Weight.replace('4s','4.04')

In [ ]:
df[['Weight','Weight_LBS']]

In [ ]:
#Operating System Typo
df['Operating System'] = df['Operating System'].replace('macOS','Mac OS')

In [ ]:
df['RAM']

In [ ]:
#Clearing GB from string and converting to int
df.RAM = df.RAM.str.replace('GB','')
df.RAM = df.RAM.astype(int)

In [ ]:
df.info()

In [ ]:
#Remove Leading Space in Column Name
df.rename(columns = {' Storage':'Storage'}, inplace = True)

In [ ]:
df['Screen Size']

In [ ]:
#Clearing " from string and converting to float
df['Screen Size'] = df['Screen Size'].str.replace('"','')
df['Screen Size'] = df['Screen Size'].astype(float)

In [ ]:
df.info()

In [ ]:
""" Pattern of the CPU string Brand, Model Number, and Speed
Since the brand and model number vary in length
The string will be flipped to take the Speed then
take the brand from the front of the string """
CPU_df = df[['CPU']].copy()
# Python code
# To reverse words in a given string
CPU_reversed = []
# input string
for x in CPU_df.CPU:
    # reversing words in a given string
    s = x.split()[::-1]
    l = []
    for i in s:
        # appending reversed words to l
        l.append(i)
    # printing reverse words
    CPU_reversed.append(" ".join(l))

In [ ]:
#Creating New feature - CPU Speed
CPU_df['CPU_reversed'] = CPU_reversed
new = CPU_df['CPU_reversed'].str.split(' ', expand=True, n=2)
df['CPU_Speed'] = new[0]
df['CPU_Speed'] = df['CPU_Speed'].str.replace('GHz','')
df['CPU_Speed'] = df['CPU_Speed'].astype(float)
new['CPU_Flipped'] = new[1] + " " +new[2]

In [ ]:
# Python code
# To reverse words in a given string
CPU_flipped = []
# input string
for x in new.CPU_Flipped:
    # reversing words in a given string
    s = x.split()[::-1]
    l = []
    for i in s:
        # appending reversed words to l
        l.append(i)
    # printing reverse words
    CPU_flipped.append(" ".join(l))

In [ ]:
#Creating New features - CPU Brand and Model
new['CPU_flipped'] = CPU_flipped
new2 = new['CPU_flipped'].str.split(' ', expand=True, n=1)
df['CPU Brand'] = new2[0]
df['CPU Model'] = new2[1]

In [ ]:
#Exploring new features
CPU_speed = df['CPU_Speed'].unique()
CPU_speed.sort()
print(df['CPU_Speed'].value_counts())
CPU_speed

In [ ]:
df.info()

In [ ]:
#Exploring new features
CPU_brand = df['CPU Brand'].unique()
CPU_brand.sort()
print(df['CPU Brand'].value_counts())
CPU_brand

In [ ]:
#Exploring new features
CPU_model = df['CPU Model'].unique()
CPU_model.sort()
print(df['CPU Model'].value_counts())
CPU_model

In [ ]:
#Creating new features - GPU Brand and Model from GPU
new3 = df['GPU'].str.split(' ', expand=True, n=1)
df['GPU Brand'] = new3[0]
df['GPU Model'] = new3[1]

In [ ]:
GPU_model = df['GPU Model'].unique()
GPU_model.sort()
print(df['GPU Model'].value_counts())
GPU_model

In [ ]:
GPU_brand = df['GPU Brand'].unique()
GPU_brand.sort()
print(df['GPU Brand'].value_counts())
GPU_brand

In [ ]:
#Some of the Model Names had redundant information captured in other columns
new4 = df['Model Name'].str.split('(', expand=True, n=1)
df['Model Name Cleaned'] = new4[0]

In [ ]:
model_name_c = df['Model Name Cleaned'].unique()
model_name_c.sort()
print(df['Model Name Cleaned'].value_counts())
model_name_c

In [ ]:
#Creating new feature of whether the screen is touchscreen
screen_touch = []

for x in df.Screen:
    if 'Touchscreen' in x:
        screen_touch.append(1)
    else:
        screen_touch.append(0)

In [ ]:
#Creating new feature of whether the screen is hd
screen_hd = []

for x in df.Screen:
    if 'HD' in x:
        screen_hd.append(1)
    else:
        screen_hd.append(0)

In [ ]:
df['Touchscreen'] = screen_touch
df['Screen_HD'] = screen_hd

In [ ]:
new5 = df['Screen'].str.split(' ', expand=True)
new5 = new5.fillna('0')

In [ ]:
#Creating new feature - screen quality from Screen column
screen_quality = []
for index,row in new5.iterrows():
    screen_quality.append(row[row.str.contains('x')].values[0])
 #       screen_quality.append(row[index])
  #  else:
   #     screen_quality.append(0)

In [ ]:
df['Screen Quality'] = screen_quality

In [ ]:
screen_quality_cleaned = df['Screen Quality'].unique()
screen_quality_cleaned.sort()
print(df['Screen Quality'].value_counts())
screen_quality_cleaned

In [ ]:
df.Screen

In [ ]:
""" Certain rows have multiple storage types and sizes.
Combining multiple storage types into total storage in the TB units """
new6 = df['Storage'].str.split(' ', expand=True)
storage = []
for index,row in new6.iterrows():
    storage.append(row[row.str.contains('B')].values)
new6 = new6.fillna('0')

In [ ]:
storage_df = pd.DataFrame(storage)

In [ ]:
storage_df = storage_df.fillna('0GB')

In [ ]:
#*(1024*1024*1024)
#*(1024*1024*1024*1024)
col_0 = []
for val in storage_df[0]:
    if 'GB' in val:
        col_0.append(int(re.findall(r'\d+',val)[0])*(1024*1024*1024))
    elif 'TB' in val:
        col_0.append(int(re.findall(r'\d+',val)[0])*(1024*1024*1024*1024))
    else:
        print('Nothing')

In [ ]:
col_1 = []
for val in storage_df[1]:
    if 'GB' in val:
        col_1.append(int(re.findall(r'\d+',val)[0])*(1024*1024*1024))
    elif 'TB' in val:
        col_1.append(int(re.findall(r'\d+',val)[0])*(1024*1024*1024*1024))
    else:
        print('Nothing')

In [ ]:
res_list = [col_0[i] + col_1[i] for i in range(len(col_0))]
total_storage_list = [x/(1024*1024*1024*1024) for x in res_list]
df['Total Storage in TB'] = total_storage_list

In [ ]:
df[['Storage','Total Storage in TB']]

In [ ]:
df.to_csv('datasets/laptops_preprocessed.csv',index=True)

In [ ]:
df = pd.read_csv('datasets/laptops_preprocessed.csv',index_col=0)
df.head()


,Manufacturer,Model Name,Model Name Cleaned,Category,Screen Size,Screen,Touchscreen,Screen_HD,Screen Quality,CPU,...,Storage,Total Storage in TB,GPU,GPU Brand,GPU Model,Operating System,Weight,Weight_LBS,Price,Price_USD
0,Apple,MacBook Pro,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,0,0,2560x1600,Intel Core i5 2.3GHz,...,128GB SSD,0.125,Intel Iris Plus Graphics 640,Intel,Iris Plus Graphics 640,Mac OS,1.37,3.020334,11912523.48,145375.19820
1,Apple,Macbook Air,Macbook Air,Ultrabook,13.3,1440x900,0,0,1440x900,Intel Core i5 1.8GHz,...,128GB Flash Storage,0.125,Intel HD Graphics 6000,Intel,HD Graphics 6000,Mac OS,1.34,2.954195,7993374.48,97547.62720
2,HP,250 G6,250 G6,Notebook,15.6,Full HD 1920x1080,0,1,1920x1080,Intel Core i5 7200U 2.5GHz,...,256GB SSD,0.250,Intel HD Graphics 620,Intel,HD Graphics 620,No OS,1.86,4.100599,5112900.00,62395.58329
3,Apple,MacBook Pro,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,0,0,2880x1800,Intel Core i7 2.7GHz,...,512GB SSD,0.500,AMD Radeon Pro 455,AMD,Radeon Pro 455,Mac OS,1.83,4.034460,22563005.40,275348.99620
4,Apple,MacBook Pro,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,0,0,2560x1600,Intel Core i5 3.1GHz,...,256GB SSD,0.250,Intel Iris Plus Graphics 650,Intel,Iris Plus Graphics 650,Mac OS,1.37,3.020334,16037611.20,195715.95480


In [ ]:
#apply(lambda x: int(x.split(' ')[0]))
#Example of cleaner way to create new features

In [ ]:
objList = df.select_dtypes(include = "object").columns
print (objList)

Index(['Manufacturer', 'Model Name', 'Model Name Cleaned', 'Category',
       'Screen', 'Screen Quality', 'CPU', 'CPU Brand', 'CPU Model', 'Storage',
       'GPU', 'GPU Brand', 'GPU Model', 'Operating System'],
      dtype='object')


In [ ]:
#Label Encoding for object to numeric conversion
le = LabelEncoder()

for feat in objList:
    df[feat] = le.fit_transform(df[feat].astype(str))

print (df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1302 entries, 0 to 1301
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Manufacturer         1302 non-null   int32  
 1   Model Name           1302 non-null   int32  
 2   Model Name Cleaned   1302 non-null   int32  
 3   Category             1302 non-null   int32  
 4   Screen Size          1302 non-null   float64
 5   Screen               1302 non-null   int32  
 6   Touchscreen          1302 non-null   int64  
 7   Screen_HD            1302 non-null   int64  
 8   Screen Quality       1302 non-null   int32  
 9   CPU                  1302 non-null   int32  
 10  CPU Brand            1302 non-null   int32  
 11  CPU Model            1302 non-null   int32  
 12  CPU_Speed            1302 non-null   float64
 13  RAM                  1302 non-null   int64  
 14  Storage              1302 non-null   int32  
 15  Total Storage in TB  1302 non-null   f

In [ ]:
#Saving df after LabelEncoder is applied
df.to_csv('datasets/laptops_afterlabelencoder.csv',index=True)

## Normality Test<a id="stats"></a>

## Splitting Data<a id="train_split"></a>

In [ ]:
#Choosing the features to be used in the model
X = df.drop(['Model Name','Screen','CPU','Storage','GPU','Weight','Price','Price_USD'],axis='columns')

In [ ]:
#Choosing target for the model
y = df.Price_USD
y.head()

0    145375.19820
1     97547.62720
2     62395.58329
3    275348.99620
4    195715.95480
Name: Price_USD, dtype: float64

In [ ]:
#Splitting train and test data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=43)

## Setting Model<a id="set"></a>

In [ ]:
#Deciding on best model for the data

regressors = [
    KNeighborsRegressor(),
    GradientBoostingRegressor(),
    KNeighborsRegressor(),
    ExtraTreesRegressor(),
    RandomForestRegressor(),
    DecisionTreeRegressor(),
    LinearRegression(),
    Lasso(),
    Ridge()
]

cv = ShuffleSplit(n_splits=5,test_size=0.2, random_state=0)
head = 10
for model in regressors[:head]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test) 
    print(model)
    print("\tExplained variance:", explained_variance_score(y_test, y_pred))
    print("\tMean absolute error:", mean_absolute_error(y_test, y_pred))
    print("\tR2 score:", r2_score(y_test, y_pred))
    print(cross_val_score(model,X,y,cv=cv))
    print()

KNeighborsRegressor()
	Training time: 0.002s
	Prediction time: 0.017s
	Explained variance: 0.7456339753457355
	Mean absolute error: 25379.263703348657
	R2 score: 0.7446258061416231

GradientBoostingRegressor()
	Training time: 0.158s
	Prediction time: 0.002s
	Explained variance: 0.8831144798210268
	Mean absolute error: 18919.316523457474
	R2 score: 0.8790154524455557

KNeighborsRegressor()
	Training time: 0.001s
	Prediction time: 0.012s
	Explained variance: 0.7456339753457355
	Mean absolute error: 25379.263703348657
	R2 score: 0.7446258061416231

ExtraTreesRegressor()
	Training time: 0.352s
	Prediction time: 0.012s
	Explained variance: 0.9031180342930277
	Mean absolute error: 16219.557399392028
	R2 score: 0.9005268273155099

RandomForestRegressor()
	Training time: 0.539s
	Prediction time: 0.015s
	Explained variance: 0.8868551479248608
	Mean absolute error: 16854.727187884182
	R2 score: 0.8856871244724549

DecisionTreeRegressor()
	Training time: 0.008s
	Prediction time: 0.002s
	Explained

## Training Model<a id="training"></a>

In [ ]:
#Hyperparameter tuning for GradientBoostingRegressor()
parameters = {'learning_rate': [0.01,0.02,0.03,0.04],
                  'subsample'    : [0.9, 0.5, 0.2, 0.1],
                  'n_estimators' : [100,500,1000, 1500],
                  'max_depth'    : [4,6,8,10]
                 }

grid_GBR = GridSearchCV(estimator=GradientBoostingRegressor(), param_grid = parameters, cv = 2, n_jobs=-1)
grid_GBR.fit(X_train, y_train)

print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",grid_GBR.best_estimator_)
print("\n The best score across ALL searched params:\n",grid_GBR.best_score_)
print("\n The best parameters across ALL searched params:\n",grid_GBR.best_params_)

In [ ]:
#Hyperparameter tuning for ExtraTreesRegressor()
param_grid = {
    'n_estimators': [10,50,100],
    'criterion': ['mse', 'mae'],
    'max_depth': [2,8,16,32,50],
    'min_samples_split': [2,4,6],
    'min_samples_leaf': [1,2],
    #'oob_score': [True, False],
    'max_features': ['auto','sqrt','log2'],    
    'bootstrap': [True, False],
    'warm_start': [True, False],
}

model = ExtraTreesRegressor ()

gcv = GridSearchCV(model,param_grid,cv=5,n_jobs=-1).fit(X_train,y_train)
print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",gcv.best_estimator_)
print("\n The best score across ALL searched params:\n",gcv.best_score_)
print("\n The best parameters across ALL searched params:\n",gcv.best_params_)

 Results from Grid Search 

 The best estimator across ALL searched params:
 ExtraTreesRegressor(criterion='mse', max_depth=50, max_features='sqrt',
                    min_samples_split=4, n_estimators=50, warm_start=True)

 The best score across ALL searched params:
 0.8254279245091919

 The best parameters across ALL searched params:
 {'bootstrap': False, 'criterion': 'mse', 'max_depth': 50, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 50, 'warm_start': True}


c:\Users\hinso\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\ensemble\_forest.py:400: FutureWarning: Criterion 'mse' was deprecated in v1.0 and will be removed in version 1.2. Use `criterion='squared_error'` which is equivalent.
  warn(


In [ ]:
#Hyperparameter tuning for RandomForestRegressor()
rfc=RandomForestRegressor(random_state=42)

param_grid = { 
    'n_estimators': [200, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth' : [4,5,6,7,8],
    'criterion' :['squared_error', 'absolute_error', 'friedman_mse', 'poisson']
}

CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)
CV_rfc.fit(X_train, y_train)

print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",CV_rfc.best_estimator_)
print("\n The best score across ALL searched params:\n",CV_rfc.best_score_)
print("\n The best parameters across ALL searched params:\n",CV_rfc.best_params_)

In [ ]:
#Hyperparameter tuning for DecisionTreeRegressor()

parameters={"splitter":["best","random"],
            "max_depth" : [1,3,5,7,9,11,12],
           "min_samples_leaf":[1,2,3,4,5,6,7,8,9,10],
           "min_weight_fraction_leaf":[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
           "max_features":["auto","log2","sqrt",None],
           "max_leaf_nodes":[None,10,20,30,40,50,60,70,80,90] }

tuning_model=GridSearchCV(DecisionTreeRegressor(),param_grid=parameters,scoring='neg_mean_squared_error',cv=3,verbose=3)
tuning_model.fit(X_train,y_train)

print(" Results from Grid Search " )
print("\n The best estimator across ALL searched params:\n",tuning_model.best_estimator_)
print("\n The best score across ALL searched params:\n",tuning_model.best_score_)
print("\n The best parameters across ALL searched params:\n",tuning_model.best_params_)

In [ ]:
#Checking scores of the hyperparameter tuned models
regressors = [
    ExtraTreesRegressor(criterion='mse', max_depth=50, max_features='sqrt',
                    min_samples_split=4, n_estimators=50, warm_start=True),
    RandomForestRegressor(max_depth=8, max_features='sqrt', n_estimators=200,
                      random_state=42)
]

head = 2
for model in regressors[:head]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test) 
    print(model)
    print("\tExplained variance:", explained_variance_score(y_test, y_pred))
    print("\tMean absolute error:", mean_absolute_error(y_test, y_pred))
    print("\tR2 score:", r2_score(y_test, y_pred))
    print(cross_val_score(model,X,y,cv=cv))
    print()

c:\Users\hinso\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\ensemble\_forest.py:400: FutureWarning: Criterion 'mse' was deprecated in v1.0 and will be removed in version 1.2. Use `criterion='squared_error'` which is equivalent.
  warn(


ExtraTreesRegressor(criterion='mse', max_depth=50, max_features='sqrt',
                    min_samples_split=4, n_estimators=50, warm_start=True)
	Training time: 0.082s
	Prediction time: 0.007s
	Explained variance: 0.8919367898716148
	Mean absolute error: 16665.30493547048
	R2 score: 0.8890409259834438

RandomForestRegressor(max_depth=8, max_features='sqrt', n_estimators=200,
                      random_state=42)
	Training time: 0.300s
	Prediction time: 0.016s
	Explained variance: 0.8486892426386257
	Mean absolute error: 19978.59634288556
	R2 score: 0.8450947501712552



## Best Results<a id="compare_results"></a>

In [ ]:
#Final model and score
best_model = ExtraTreesRegressor(criterion='mse', max_depth=50, max_features='sqrt',
                    min_samples_split=4, n_estimators=50, warm_start=True)

best_model.fit(X_train,y_train)
best_model.score(X_test,y_test)

c:\Users\hinso\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\ensemble\_forest.py:400: FutureWarning: Criterion 'mse' was deprecated in v1.0 and will be removed in version 1.2. Use `criterion='squared_error'` which is equivalent.
  warn(


0.8815112581250318

## Finalizing Workflow<a id="workflow"></a>

## Fitting the final model<a id="fit"></a>

## Interface(Streamlit)<a id="interface"></a>

## Saving Files<a id="store"></a>

In [ ]:
#Saving the model
with open('laptop_price_regression.pickle','wb') as f:
    pickle.dump(best_model,f)

In [ ]:
#Saving Column names
columns={
    'data_columns': [col.lower() for col in X.columns]
}

with open('columns.json','w') as f:
    f.write(json.dumps(columns))

In [ ]:
#Storing LabelEncoder for GUI use
filehandler = open("le.obj","wb")
pickle.dump(le,filehandler)
filehandler.close()

In [ ]:
#Saving unique column values for GUI drop down fields
dict = {}

for (columnName, columnData) in df.items():
    #print('Column Name : ', columnName)
    #print('Column Contents : ', columnData.unique())
    dict[columnName] = columnData.unique().tolist()

with open("column_values.json", "w") as outfile:
    json.dump(dict, outfile)


Column Name :  Manufacturer
Column Name :  Model Name
Column Name :  Model Name Cleaned
Column Name :  Category
Column Name :  Screen Size
Column Name :  Screen
Column Name :  Touchscreen
Column Name :  Screen_HD
Column Name :  Screen Quality
Column Name :  CPU
Column Name :  CPU Brand
Column Name :  CPU Model
Column Name :  CPU_Speed
Column Name :  RAM
Column Name :  Storage
Column Name :  Total Storage in TB
Column Name :  GPU
Column Name :  GPU Brand
Column Name :  GPU Model
Column Name :  Operating System
Column Name :  Weight
Column Name :  Weight_LBS
Column Name :  Price
Column Name :  Price_USD


C:\Users\hinso\AppData\Local\Temp\ipykernel_5104\3705546538.py:3: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for (columnName, columnData) in df.iteritems():


## Conclusion<a id="conclusion"></a>